# 🤖 Getting Started with the OpenAI API
### A Beginner's Guide

Welcome! This notebook will walk you through everything you need to start using the OpenAI API — from understanding what an API is, to calling AI models, generating embeddings, and using powerful tools like function calling.

---

**What you'll learn:**

1. 🌐 What is an API and how does it work?
2. 🔑 Setting up your API key
3. 💬 Chat Completions — talking to AI
4. 🧠 Prompting, Models & Reasoning
5. 🔢 Embeddings & Semantic Search
6. 🛠️ Function Calling / Tool Use

---

> **Prerequisites:** Basic Python knowledge. No AI experience needed!


## 📦 Install Dependencies

First, let's install the official OpenAI Python library.

`!pip install openai numpy scikit-learn`

---
## 🌐 Section 1: What is an API?

**API** stands for **Application Programming Interface**. Think of it like a restaurant:

| Restaurant | API |
|---|---|
| You (customer) | Your Python code |
| Waiter | The API |
| Kitchen | OpenAI's servers |
| Menu | API documentation |
| Food | The AI's response |

You place an **order (request)** → the waiter (API) takes it to the kitchen (OpenAI) → the kitchen prepares your food (runs AI) → the waiter brings back your food **(response)**.

### How HTTP Requests Work

Under the hood, every API call is an **HTTP request** — the same technology your browser uses to load websites.

```
Your Code  ──── POST /v1/chat/completions ────▶  OpenAI Server
               { model, messages, ... }                │
                                                       ▼
Your Code  ◀─── 200 OK ─────────────────────  AI generates response
               { choices: [{ message: ... }] }
```

The OpenAI library handles all this complexity for you!

---

## 🔑 Section 2: Setting Up Your API Key

To use the OpenAI API, you need an **API key** — a secret credential that identifies you.

### How to get your API key:
1. Go to [platform.openai.com](https://platform.openai.com)
2. Sign up or log in
3. Navigate to **API Keys** in the dashboard
4. Click **"Create new secret key"**
5. Copy it immediately — you won't see it again!


In [ ]:
import os
from openai import OpenAI

# Initialize the OpenAI client

OPENAI_API_KEY=...
client = OpenAI(api_key=OPENAI_API_KEY)

print("✅ OpenAI client initialized successfully!")

<div align="left" style="border: 1px solid #ccc; padding: 15px; border-radius: 6px; background-color: #f6f8fa;">

⚠️ **Security Rules for API Keys**

- **Never** hardcode your key in code you share or commit to GitHub
- **Never** share it publicly
- Use environment variables or a `.env` file
</div>

### Best Practice: Load from a `.env` file

Hardcoding secrets can be prone to accidental sharing or commits, especially when you are using source code versioning tools like Github. To prevent this, we often

Create a file called `.env` in the same folder as this notebook:
```
OPENAI_API_KEY=sk-your-key-here
```
Then load it with:

In [ ]:
# Optional: uncomment if using a .env file
# !pip install python-dotenv
# from dotenv import load_dotenv
# load_dotenv()  # loads OPENAI_API_KEY from .env automatically

---

## 💬 Section 3: Chat Completions

The **Chat Completions** endpoint is the core of the OpenAI API. It lets you send messages and receive AI-generated replies — just like a chat interface.

### The Message Format

Messages follow a simple structure with **roles**:

| Role | Who | Purpose |
|---|---|---|
| `system` | AI's instructions | Sets behavior, persona, constraints |
| `user` | Human | The actual question or request |
| `assistant` | AI's past reply | Used for multi-turn conversations |

### Your First API Call

In [ ]:
# Your first chat completion!
response = client.chat.completions.create(
    model="gpt-4o-mini",           # The model to use
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant who explains things simply."
        },
        {
            "role": "user",
            "content": "What is machine learning in one sentence?"
        }
    ]
)

# Extract the text from the response
reply = response.choices[0].message.content
print("AI says:", reply)

### Understanding the Response Object

The API returns a rich object. Let's explore it:

In [ ]:
# Inspect the full response
print("Model used:      ", response.model)
print("Stop reason:     ", response.choices[0].finish_reason)
print("Prompt tokens:   ", response.usage.prompt_tokens)
print("Response tokens: ", response.usage.completion_tokens)
print("Total tokens:    ", response.usage.total_tokens)

# 💡 Tokens are roughly 0.75 words each.
# Token usage determines your cost — shorter prompts = cheaper!

### Multi-Turn Conversations

The API is **stateless** — it has no memory between calls. To have a conversation, you must include the full history each time.

In [ ]:
def chat(messages):
    """Send a conversation and return the assistant's reply."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )
    return response.choices[0].message.content


# Build a conversation turn by turn
conversation = [
    {"role": "system", "content": "You are a friendly cooking assistant."}
]

# Turn 1
user_msg_1 = "I want to make pasta. What do I need?"
conversation.append({"role": "user", "content": user_msg_1})
reply_1 = chat(conversation)
conversation.append({"role": "assistant", "content": reply_1})
print("User:", user_msg_1)
print("AI:", reply_1)
print()

# Turn 2 — AI remembers the context!
user_msg_2 = "I don't have eggs. Can I substitute something?"
conversation.append({"role": "user", "content": user_msg_2})
reply_2 = chat(conversation)
print("User:", user_msg_2)
print("AI:", reply_2)

---

## 🧠 Section 4: Prompting, Models & Reasoning

### 4.1 Choosing a Model

OpenAI offers several models with different trade-offs:

| Model | Speed | Cost | Best For |
|---|---|---|---|
| `gpt-4o-mini` | ⚡ Fast | 💰 Cheap | Simple tasks, high volume |
| `gpt-4o` | 🚀 Fast | 💰💰 Moderate | Complex tasks, vision |
| `o1-mini` | 🐢 Slower | 💰💰 Moderate | Math, coding, reasoning |
| `o1` | 🐢 Slower | 💰💰💰 Expensive | Deep reasoning, hard problems |

> **Rule of thumb:** Start with `gpt-4o-mini`. Upgrade to stronger models only if quality isn't sufficient.

### 4.2 Key Parameters

In [ ]:
# Exploring key parameters

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Give me a creative name for a coffee shop."}
    ],
    temperature=1.2,     # 0.0 = deterministic/focused, 2.0 = very creative/random
    max_tokens=50,       # Limit response length (controls cost too)
    n=3,                 # Get multiple responses at once
)

print("3 creative coffee shop names:")
for i, choice in enumerate(response.choices):
    print(f"  {i+1}. {choice.message.content.strip()}")

### 4.3 Prompting Techniques

The quality of your **prompt** (what you send the AI) directly determines the quality of the response. Here are key techniques:

In [ ]:
# ── Technique 1: Zero-Shot Prompting ─────────────────────────────────────────
# Just ask directly — no examples given

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Classify this review as Positive, Neutral, or Negative: 'The food was okay but the service was terrible.'"}]
)
print("Zero-shot:", response.choices[0].message.content.strip())

In [ ]:
# ── Technique 2: Few-Shot Prompting ──────────────────────────────────────────
# Provide examples to guide the model's behavior

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": """Classify sentiment. Examples:
"I love this product!" → Positive
"It broke after one day" → Negative  
"It arrived on time" → Neutral

Now classify: "The battery life is decent but the screen is blurry"""""
        }
    ]
)
print("Few-shot:", response.choices[0].message.content.strip())

In [ ]:
# ── Technique 3: Chain-of-Thought Prompting ───────────────────────────────────
# Ask the model to reason step-by-step before answering

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": """
A store sells apples for $0.50 each and oranges for $0.75 each.
Maria buys 4 apples and 3 oranges. She pays with a $5 bill.
How much change does she get?

Think step by step.
"""
        }
    ]
)
print(response.choices[0].message.content)

In [ ]:
# ── Technique 4: Persona + Format Control ────────────────────────────────────
# Use the system prompt to control style and output format

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": """You are a senior software engineer. 
Always respond in this exact format:
ANSWER: <one sentence answer>
EXAMPLE: <a code snippet>
TIP: <one practical tip>"""
        },
        {
            "role": "user",
            "content": "What is a Python list comprehension?"
        }
    ]
)
print(response.choices[0].message.content)

### 4.4 Structured Output (JSON Mode)

You can force the model to always return valid JSON — perfect for programmatic use.

In [ ]:
import json

response = client.chat.completions.create(
    model="gpt-4o-mini",
    response_format={"type": "json_object"},  # Enforce JSON output
    messages=[
        {
            "role": "system",
            "content": "You extract information and return it as JSON."
        },
        {
            "role": "user",
            "content": """Extract the person info from this text and return JSON with keys: name, age, city.
Text: 'Hi, I'm Juan, a 28-year-old developer from Manila.'"""
        }
    ]
)

data = json.loads(response.choices[0].message.content)
print("Parsed JSON:", data)
print("Name:", data["name"])
print("City:", data["city"])

---

## 🔢 Section 5: Embeddings & Semantic Search

### What are Embeddings?

An **embedding** is a list of numbers (a vector) that represents the *meaning* of text. Texts with similar meanings will have vectors that are close together in space.

```
"dog"        →  [0.12, -0.83, 0.45, ...]  ─┐
"puppy"      →  [0.11, -0.80, 0.43, ...]  ─┘  ← These are CLOSE
"quantum"    →  [-0.92, 0.33, -0.11, ...] ─── Far away
```

This lets you find documents that are *semantically similar* — even if they use different words.

In [ ]:
import numpy as np

def get_embedding(text, model="text-embedding-3-small"):
    """Get the embedding vector for a piece of text."""
    response = client.embeddings.create(
        input=text,
        model=model
    )
    return np.array(response.data[0].embedding)

# Try it out
embedding = get_embedding("Hello, world!")
print(f"Embedding dimensions: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")
print(f"\nEach text becomes a {len(embedding)}-dimensional vector!")

In [ ]:
def cosine_similarity(a, b):
    """Measure how similar two vectors are. Returns 0 (different) to 1 (identical)."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Compare similarity between phrases
phrases = [
    "I love programming in Python",
    "Python is my favorite coding language",
    "I enjoy cooking Italian food",
    "Pasta and pizza are delicious",
]

print("Getting embeddings...")
embeddings = [get_embedding(p) for p in phrases]

print("\nSimilarity scores (0 = different, 1 = identical):")
for i in range(len(phrases)):
    for j in range(i+1, len(phrases)):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f"  {sim:.3f} | '{phrases[i][:35]}...' vs '{phrases[j][:35]}...'") 

### Building a Simple Semantic Search Engine

In [ ]:
# A tiny knowledge base
documents = [
    "Python is a high-level programming language known for its simplicity.",
    "Machine learning is a subset of AI that learns from data.",
    "The Eiffel Tower is located in Paris, France.",
    "Neural networks are inspired by the structure of the human brain.",
    "JavaScript is widely used for web development and frontend programming.",
    "Deep learning uses multiple layers of neural networks for complex tasks.",
    "The Great Wall of China stretches over 13,000 miles.",
    "APIs allow different software applications to communicate with each other.",
]

print("Indexing documents...")
doc_embeddings = [get_embedding(doc) for doc in documents]
print(f"✅ Indexed {len(documents)} documents")


def semantic_search(query, top_k=3):
    """Find the most relevant documents for a query."""
    query_embedding = get_embedding(query)
    
    scores = [
        (cosine_similarity(query_embedding, doc_emb), doc)
        for doc_emb, doc in zip(doc_embeddings, documents)
    ]
    scores.sort(reverse=True)
    
    print(f"\n🔍 Query: '{query}'")
    print("Top results:")
    for rank, (score, doc) in enumerate(scores[:top_k], 1):
        print(f"  {rank}. [{score:.3f}] {doc}")


semantic_search("How do deep learning systems work?")
semantic_search("Famous landmarks in Europe")

---

## 🛠️ Section 6: Function Calling / Tool Use

**Function calling** lets the AI decide when to call your custom functions to get real data — like checking the weather, querying a database, or calling an external API.

### How it Works

```
1. You tell the AI about your functions (name, description, parameters)
2. User asks a question
3. AI decides IF and WHICH function to call, with WHAT arguments
4. Your code runs the function and gets the result
5. You send the result back to the AI
6. AI gives a final natural language answer
```

In [ ]:
import json

# ── Step 1: Define your real Python functions ─────────────────────────────────

def get_weather(city: str) -> dict:
    """Simulated weather API (replace with a real one!)."""
    # In real life, you'd call an actual weather API here
    weather_data = {
        "Manila":    {"temp": 32, "condition": "Sunny",  "humidity": 75},
        "Tokyo":     {"temp": 18, "condition": "Cloudy", "humidity": 60},
        "London":    {"temp": 12, "condition": "Rainy",  "humidity": 85},
        "New York":  {"temp": 22, "condition": "Partly cloudy", "humidity": 55},
    }
    return weather_data.get(city, {"temp": 20, "condition": "Unknown", "humidity": 50})


def get_stock_price(ticker: str) -> dict:
    """Simulated stock price lookup."""
    prices = {"AAPL": 189.50, "GOOGL": 141.20, "MSFT": 378.85, "NVDA": 875.40}
    price = prices.get(ticker.upper(), None)
    if price:
        return {"ticker": ticker.upper(), "price": price, "currency": "USD"}
    return {"error": f"Ticker '{ticker}' not found"}


# ── Step 2: Describe your functions to the AI ─────────────────────────────────

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city, e.g. 'Manila'"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "Get the current stock price for a given ticker symbol",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {
                        "type": "string",
                        "description": "Stock ticker symbol, e.g. 'AAPL'"
                    }
                },
                "required": ["ticker"]
            }
        }
    }
]

print("✅ Tools defined!")

In [ ]:
# ── Step 3: The full function-calling loop ────────────────────────────────────

# Map function names to actual Python functions
available_functions = {
    "get_weather": get_weather,
    "get_stock_price": get_stock_price,
}


def run_with_tools(user_message):
    """Run a conversation that can call real tools."""
    print(f"\n👤 User: {user_message}")
    
    messages = [{"role": "user", "content": user_message}]
    
    # First call — AI may decide to use a tool
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
        tool_choice="auto"  # Let AI decide when to use tools
    )
    
    ai_message = response.choices[0].message
    
    # Check if AI wants to call a tool
    if ai_message.tool_calls:
        messages.append(ai_message)  # Add AI's tool request to history
        
        # Execute each tool the AI requested
        for tool_call in ai_message.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)
            
            print(f"🔧 AI is calling: {fn_name}({fn_args})")
            
            # Actually run the function
            result = available_functions[fn_name](**fn_args)
            
            # Add the tool result to the conversation
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })
        
        # Second call — AI uses the tool results to form a final answer
        final_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )
        print("🤖 AI:", final_response.choices[0].message.content)
    else:
        # AI answered directly without tools
        print("🤖 AI:", ai_message.content)


# Test it!
run_with_tools("What's the weather like in Tokyo right now?")
run_with_tools("What is Apple's stock price?")
run_with_tools("What is 2 + 2?")  # No tool needed for this one

---

## 🎉 Summary & Next Steps

Congratulations! You've now learned the core building blocks of the OpenAI API:

| Topic | Key Takeaway |
|---|---|
| 🌐 APIs | Request → Server → Response; OpenAI handles the complexity |
| 🔑 Authentication | Always use environment variables for API keys |
| 💬 Chat Completions | Use roles (system/user/assistant) to structure conversations |
| 🧠 Prompting | Zero-shot, few-shot, chain-of-thought, JSON mode |
| 🔢 Embeddings | Text → numbers → semantic similarity search |
| 🛠️ Function Calling | Let AI call your real-world functions intelligently |

### 🚀 Additional References

- 📖 [OpenAI API Documentation](https://platform.openai.com/docs)
- 🏫 [OpenAI Cookbook](https://cookbook.openai.com) — practical examples

---

> **💡 Pro Tip:** Always monitor your token usage and set spending limits on your OpenAI account dashboard to avoid surprise bills while learning!